# 🇦🇪 MallVision AI — Emirati Classifier Training (YOLOv8)
This notebook trains a custom YOLOv8 classification model to detect **Emirati Traditional Attire** vs **Western/Casual Attire** with gender split:
- `male_emirati` (Kandura / Ghutra)
- `female_emirati` (Abaya / Niqab / Hijab)
- `male_non_emirati` (Polo / Casual / Western)
- `female_non_emirati` (Dress / Casual / Western)

### Instructions:
1. In the menu: **Runtime → Change runtime type → T4 GPU**
2. Run all cells (`Ctrl + F9`)
3. At Cell 2, upload your `emirati_dataset.zip` (or place it in Google Drive)
4. When training finishes, `emirati_model_best.pt` will automatically download to your computer!

In [ ]:
# Cell 1: Install Ultralytics
!pip install ultralytics -q

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Extract or Upload Dataset
import os, zipfile, shutil
from google.colab import files

if not os.path.exists("dataset"):
    if os.path.exists("/content/drive/MyDrive/emirati_dataset.zip"):
        shutil.copy("/content/drive/MyDrive/emirati_dataset.zip", "emirati_dataset.zip")
    
    if os.path.exists("emirati_dataset.zip"):
        print("Extracting existing emirati_dataset.zip...")
        with zipfile.ZipFile("emirati_dataset.zip", 'r') as zip_ref:
            zip_ref.extractall("dataset")
    else:
        print("Upload your 'emirati_dataset.zip' file:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.zip'):
                with zipfile.ZipFile(fn, 'r') as zip_ref:
                    zip_ref.extractall("dataset")

print("\nClasses found:")
if os.path.exists("dataset/train"):
    for c in sorted(os.listdir("dataset/train")):
        count = len(os.listdir(os.path.join("dataset/train", c)))
        print(f"  - {c}: {count} training images")

In [ ]:
# Cell 3: Train YOLOv8 Classification Model
from ultralytics import YOLO

model = YOLO("yolov8s-cls.pt")

results = model.train(
    data="dataset",
    epochs=50,
    imgsz=224,
    batch=32,
    patience=10,
    dropout=0.3,
    augment=True,
    device=0,
    project="emirati",
    name="v1"
)

In [ ]:
# Cell 4: Download emirati_model_best.pt
import os, shutil
from google.colab import files

best_weights = "emirati/v1/weights/best.pt"
target_name = "emirati_model_best.pt"

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_name)
    print(f"✅ Training finished! Best model saved to {target_name}.")
    print("Starting browser download now...")
    files.download(target_name)
else:
    print(f"Model file not found at {best_weights}")